# Deploy RDS PostgreSQL Database for AgenticAI Demo Platform

Creates an RDS PostgreSQL instance with the same schema and seed data as the DynamoDB version.

**Tables:**
| Table | Description |
|-------|-------------|
| `sports` | Available sports |
| `events` | Sporting events (FK to sports) |
| `odds` | Odds per event per market selection |
| `accounts` | Customer profiles + balances |
| `bets` | Placed bets (FK to accounts, events) |
| `transactions` | Deposits, withdrawals, settlements (FK to accounts) |
| `promotions` | Promotional offers |
| `support_tickets` | Customer support cases (FK to accounts) |

<hr>

In [ ]:
import boto3
import json
import time

REGION = "us-east-1"
ACCOUNT_ID = "869795389844"

DB_INSTANCE_ID = "agenticai-demo-db"
DB_NAME = "agenticai"
DB_USERNAME = "agenticaiadmin"
DB_PASSWORD = "DemoPassword123!"  # For demo only
DB_INSTANCE_CLASS = "db.t3.micro"
SECURITY_GROUP_NAME = "agenticai-demo-rds-sg"

rds = boto3.client("rds", region_name=REGION)
ec2 = boto3.client("ec2", region_name=REGION)

print("Configuration ready")

## Step 1: Create VPC and Security Group\n<hr>

In [ ]:
VPC_NAME = "AgenticAIDemoVPC"
VPC_CIDR = "10.0.0.0/16"

# Check if VPC already exists
existing = ec2.describe_vpcs(Filters=[{"Name": "tag:Name", "Values": [VPC_NAME]}])
if existing["Vpcs"]:
    vpc_id = existing["Vpcs"][0]["VpcId"]
    print(f"VPC already exists: {vpc_id}")
else:
    vpc = ec2.create_vpc(CidrBlock=VPC_CIDR)
    vpc_id = vpc["Vpc"]["VpcId"]
    ec2.get_waiter("vpc_available").wait(VpcIds=[vpc_id])
    ec2.create_tags(Resources=[vpc_id], Tags=[{"Key": "Name", "Value": VPC_NAME}])
    # Enable DNS support
    ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={"Value": True})
    ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={"Value": True})
    print(f"Created VPC: {vpc_id} ({VPC_NAME})")

# Create subnets in 2 AZs (required for RDS subnet group)
azs = ec2.describe_availability_zones(State="available")["AvailabilityZones"]
az_names = [az["ZoneName"] for az in azs[:2]]

subnet_ids = []
subnet_cidrs = ["10.0.1.0/24", "10.0.2.0/24"]
existing_subnets = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpc_id]}])["Subnets"]

if len(existing_subnets) >= 2:
    subnet_ids = [s["SubnetId"] for s in existing_subnets[:2]]
    print(f"Using existing subnets: {subnet_ids}")
else:
    for i, (az, cidr) in enumerate(zip(az_names, subnet_cidrs)):
        sub = ec2.create_subnet(VpcId=vpc_id, CidrBlock=cidr, AvailabilityZone=az)
        sid = sub["Subnet"]["SubnetId"]
        ec2.create_tags(Resources=[sid], Tags=[{"Key": "Name", "Value": f"{VPC_NAME}-subnet-{i+1}"}])
        subnet_ids.append(sid)
        print(f"  Created subnet: {sid} in {az} ({cidr})")

# Create DB subnet group
DB_SUBNET_GROUP = "agenticai-demo-subnet-group"
try:
    rds.create_db_subnet_group(
        DBSubnetGroupName=DB_SUBNET_GROUP,
        DBSubnetGroupDescription="Subnets for AgenticAI demo RDS",
        SubnetIds=subnet_ids,
    )
    print(f"Created DB subnet group: {DB_SUBNET_GROUP}")
except rds.exceptions.DBSubnetGroupAlreadyExistsFault:
    rds.modify_db_subnet_group(
        DBSubnetGroupName=DB_SUBNET_GROUP,
        SubnetIds=subnet_ids,
    )
    print(f"DB subnet group already exists, updated: {DB_SUBNET_GROUP}")

# Create security group in the new VPC
try:
    sg = ec2.create_security_group(
        GroupName=SECURITY_GROUP_NAME,
        Description="Allow PostgreSQL access for AgenticAI demo",
        VpcId=vpc_id,
    )
    sg_id = sg["GroupId"]
    print(f"Created security group: {sg_id}")
except ec2.exceptions.ClientError as e:
    if "InvalidGroup.Duplicate" in str(e):
        sgs = ec2.describe_security_groups(
            Filters=[
                {"Name": "group-name", "Values": [SECURITY_GROUP_NAME]},
                {"Name": "vpc-id", "Values": [vpc_id]},
            ]
        )
        sg_id = sgs["SecurityGroups"][0]["GroupId"]
        print(f"Security group already exists: {sg_id}")
    else:
        raise

# Allow inbound PostgreSQL from within the VPC only
try:
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[{
            "IpProtocol": "tcp",
            "FromPort": 5432,
            "ToPort": 5432,
            "IpRanges": [{"CidrIp": VPC_CIDR, "Description": "PostgreSQL from VPC"}],
        }],
    )
    print(f"Added inbound rule for port 5432 from {VPC_CIDR}")
except ec2.exceptions.ClientError as e:
    if "InvalidPermission.Duplicate" in str(e):
        print("Inbound rule already exists")
    else:
        raise

print(f"\nVPC: {vpc_id} ({VPC_NAME})")
print(f"Subnets: {subnet_ids}")
print(f"Security Group: {sg_id}")

## Step 2: Create RDS PostgreSQL Instance\n<hr>

In [ ]:
# Delete existing instance if present
try:
    rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE_ID)
    print(f"Deleting existing instance: {DB_INSTANCE_ID}")
    rds.delete_db_instance(
        DBInstanceIdentifier=DB_INSTANCE_ID,
        SkipFinalSnapshot=True,
        DeleteAutomatedBackups=True,
    )
    waiter = rds.get_waiter("db_instance_deleted")
    waiter.wait(DBInstanceIdentifier=DB_INSTANCE_ID)
    print("  Deleted.")
except rds.exceptions.DBInstanceNotFoundFault:
    pass

print(f"Creating RDS instance: {DB_INSTANCE_ID}")
rds.create_db_instance(
    DBInstanceIdentifier=DB_INSTANCE_ID,
    DBName=DB_NAME,
    Engine="postgres",
    EngineVersion="16.4",
    DBInstanceClass=DB_INSTANCE_CLASS,
    MasterUsername=DB_USERNAME,
    MasterUserPassword=DB_PASSWORD,
    AllocatedStorage=20,
    StorageType="gp3",
    VpcSecurityGroupIds=[sg_id],
    DBSubnetGroupName=DB_SUBNET_GROUP,
    PubliclyAccessible=False,
    BackupRetentionPeriod=0,
    Tags=[{"Key": "Project", "Value": "AgenticAIDemo"}],
)

print("Waiting for instance to become available (this takes 5-10 minutes)...")
waiter = rds.get_waiter("db_instance_available")
waiter.wait(DBInstanceIdentifier=DB_INSTANCE_ID)

db_info = rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE_ID)
db_endpoint = db_info["DBInstances"][0]["Endpoint"]["Address"]
db_port = db_info["DBInstances"][0]["Endpoint"]["Port"]

print(f"\nRDS Instance ready!")
print(f"  Endpoint: {db_endpoint}")
print(f"  Port: {db_port}")
print(f"  Database: {DB_NAME}")

## Step 3: Install psycopg2 and Connect\n<hr>

In [ ]:
!pip install psycopg2-binary --quiet

import psycopg2

conn = psycopg2.connect(
    host=db_endpoint,
    port=db_port,
    dbname=DB_NAME,
    user=DB_USERNAME,
    password=DB_PASSWORD,
)
conn.autocommit = True
cur = conn.cursor()
print("Connected to PostgreSQL")

## Step 4: Create Schema\n<hr>

In [ ]:
cur.execute("""

-- Drop existing tables (reverse FK order)
DROP TABLE IF EXISTS support_tickets CASCADE;
DROP TABLE IF EXISTS transactions CASCADE;
DROP TABLE IF EXISTS bets CASCADE;
DROP TABLE IF EXISTS odds CASCADE;
DROP TABLE IF EXISTS events CASCADE;
DROP TABLE IF EXISTS promotions CASCADE;
DROP TABLE IF EXISTS accounts CASCADE;
DROP TABLE IF EXISTS sports CASCADE;

-- Sports
CREATE TABLE sports (
    id VARCHAR(50) PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    active_events INTEGER DEFAULT 0
);

-- Events
CREATE TABLE events (
    id VARCHAR(50) PRIMARY KEY,
    sport_id VARCHAR(50) NOT NULL REFERENCES sports(id),
    home VARCHAR(100) NOT NULL,
    away VARCHAR(100) NOT NULL,
    league VARCHAR(100),
    kickoff TIMESTAMPTZ,
    status VARCHAR(20) DEFAULT 'upcoming'
);
CREATE INDEX idx_events_sport ON events(sport_id);
CREATE INDEX idx_events_status ON events(status);

-- Odds (one row per event + market selection)
CREATE TABLE odds (
    event_id VARCHAR(50) NOT NULL REFERENCES events(id),
    selection VARCHAR(50) NOT NULL,
    odds DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (event_id, selection)
);

-- Accounts
CREATE TABLE accounts (
    account_id VARCHAR(50) PRIMARY KEY,
    email VARCHAR(200) UNIQUE,
    name VARCHAR(200),
    balance DECIMAL(12,2) DEFAULT 0.00,
    currency VARCHAR(10) DEFAULT 'USD',
    tier VARCHAR(20) DEFAULT 'bronze',
    kyc_status VARCHAR(20) DEFAULT 'pending',
    registered_at TIMESTAMPTZ,
    last_login_at TIMESTAMPTZ,
    deposit_limit DECIMAL(12,2),
    weekly_bet_limit DECIMAL(12,2),
    self_excluded BOOLEAN DEFAULT FALSE
);
CREATE INDEX idx_accounts_email ON accounts(email);

-- Bets
CREATE TABLE bets (
    bet_id VARCHAR(50) PRIMARY KEY,
    account_id VARCHAR(50) NOT NULL REFERENCES accounts(account_id),
    event_id VARCHAR(50) NOT NULL REFERENCES events(id),
    selection VARCHAR(50) NOT NULL,
    odds DECIMAL(10,2) NOT NULL,
    stake DECIMAL(12,2) NOT NULL,
    potential_return DECIMAL(12,2) NOT NULL,
    status VARCHAR(20) DEFAULT 'active',
    placed_at TIMESTAMPTZ DEFAULT NOW()
);
CREATE INDEX idx_bets_account ON bets(account_id);

-- Transactions
CREATE TABLE transactions (
    transaction_id VARCHAR(50) PRIMARY KEY,
    account_id VARCHAR(50) NOT NULL REFERENCES accounts(account_id),
    type VARCHAR(30) NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    method VARCHAR(50),
    status VARCHAR(20) DEFAULT 'pending',
    created_at TIMESTAMPTZ DEFAULT NOW(),
    description TEXT
);
CREATE INDEX idx_transactions_account ON transactions(account_id);

-- Promotions
CREATE TABLE promotions (
    id VARCHAR(50) PRIMARY KEY,
    name VARCHAR(200) NOT NULL,
    description TEXT,
    min_deposit DECIMAL(12,2),
    min_odds DECIMAL(10,2),
    min_legs INTEGER,
    event_id VARCHAR(50),
    expires DATE
);

-- Support Tickets
CREATE TABLE support_tickets (
    ticket_id VARCHAR(50) PRIMARY KEY,
    account_id VARCHAR(50) NOT NULL REFERENCES accounts(account_id),
    subject VARCHAR(500) NOT NULL,
    description TEXT,
    status VARCHAR(20) DEFAULT 'open',
    priority VARCHAR(20) DEFAULT 'medium',
    category VARCHAR(50) DEFAULT 'general',
    created_at TIMESTAMPTZ DEFAULT NOW(),
    updated_at TIMESTAMPTZ DEFAULT NOW()
);
CREATE INDEX idx_tickets_account ON support_tickets(account_id);

""")

print("Schema created: 8 tables with indexes and foreign keys")

## Step 5: Seed Data\n<hr>

In [ ]:
# --- Sports ---
cur.executemany(
    "INSERT INTO sports (id, name, active_events) VALUES (%s, %s, %s)",
    [
        ('football', 'Football', 24),
        ('basketball', 'Basketball', 18),
        ('tennis', 'Tennis', 32),
        ('horse_racing', 'Horse Racing', 12),
        ('esports', 'Esports', 8),
    ]
)
print("Seeded 5 sports")

# --- Events ---
cur.executemany(
    "INSERT INTO events (id, sport_id, home, away, league, kickoff, status) VALUES (%s, %s, %s, %s, %s, %s, %s)",
    [
        ('evt-001', 'football', 'Arsenal', 'Chelsea', 'Premier League', '2026-04-25T15:00:00Z', 'upcoming'),
        ('evt-002', 'football', 'Barcelona', 'Real Madrid', 'La Liga', '2026-04-25T20:00:00Z', 'upcoming'),
        ('evt-003', 'football', 'Bayern Munich', 'Dortmund', 'Bundesliga', '2026-04-26T17:30:00Z', 'upcoming'),
        ('evt-010', 'basketball', 'Lakers', 'Celtics', 'NBA', '2026-04-25T19:30:00Z', 'live'),
        ('evt-011', 'basketball', 'Warriors', 'Bucks', 'NBA', '2026-04-26T21:00:00Z', 'upcoming'),
        ('evt-020', 'tennis', 'Djokovic', 'Alcaraz', 'ATP Masters', '2026-04-25T14:00:00Z', 'live'),
        ('evt-021', 'tennis', 'Sinner', 'Medvedev', 'ATP Masters', '2026-04-26T16:00:00Z', 'upcoming'),
        ('evt-030', 'horse_racing', 'Ascot - 3:15 Race', '8 runners', 'Royal Ascot', '2026-04-25T15:15:00Z', 'upcoming'),
        ('evt-040', 'esports', 'T1', 'Gen.G', 'LCK', '2026-04-25T12:00:00Z', 'live'),
    ]
)
print("Seeded 9 events")

In [ ]:
# --- Odds (normalized: one row per event + selection) ---
cur.executemany(
    "INSERT INTO odds (event_id, selection, odds) VALUES (%s, %s, %s)",
    [
        # Football
        ('evt-001', 'homeWin', 2.10), ('evt-001', 'draw', 3.40), ('evt-001', 'awayWin', 3.20),
        ('evt-002', 'homeWin', 1.85), ('evt-002', 'draw', 3.60), ('evt-002', 'awayWin', 4.00),
        ('evt-003', 'homeWin', 1.70), ('evt-003', 'draw', 3.80), ('evt-003', 'awayWin', 4.50),
        # Basketball
        ('evt-010', 'homeWin', 1.90), ('evt-010', 'awayWin', 1.95), ('evt-010', 'overTotal', 1.85), ('evt-010', 'underTotal', 2.00),
        ('evt-011', 'homeWin', 2.20), ('evt-011', 'awayWin', 1.70), ('evt-011', 'overTotal', 1.90), ('evt-011', 'underTotal', 1.95),
        # Tennis
        ('evt-020', 'homeWin', 1.60), ('evt-020', 'awayWin', 2.30),
        ('evt-021', 'homeWin', 1.75), ('evt-021', 'awayWin', 2.10),
        # Horse Racing
        ('evt-030', 'runner1', 3.50), ('evt-030', 'runner2', 5.00), ('evt-030', 'runner3', 7.00), ('evt-030', 'runner4', 8.50), ('evt-030', 'runner5', 12.00),
        # Esports
        ('evt-040', 'homeWin', 1.45), ('evt-040', 'awayWin', 2.70),
    ]
)
print("Seeded 27 odds rows")

In [ ]:
# --- Accounts ---
cur.executemany(
    """INSERT INTO accounts (account_id, email, name, balance, currency, tier, kyc_status,
       registered_at, last_login_at, deposit_limit, weekly_bet_limit, self_excluded)
       VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
    [
        ('default', 'john.doe@example.com', 'John Doe', 1000.00, 'USD', 'gold', 'verified',
         '2025-11-15T09:00:00Z', '2026-04-23T08:30:00Z', 5000.00, 2000.00, False),
        ('cust-002', 'maria.garcia@example.com', 'Maria Garcia', 250.00, 'USD', 'silver', 'verified',
         '2026-01-20T14:00:00Z', '2026-04-22T19:15:00Z', 2000.00, 1000.00, False),
        ('cust-003', 'alex.new@example.com', 'Alex Thompson', 200.00, 'USD', 'bronze', 'pending',
         '2026-04-22T10:00:00Z', '2026-04-23T07:45:00Z', 1000.00, 500.00, False),
    ]
)
print("Seeded 3 accounts")

# --- Promotions ---
cur.executemany(
    "INSERT INTO promotions (id, name, description, min_deposit, min_odds, min_legs, event_id, expires) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
    [
        ('promo-001', 'Welcome Bonus', '100% match on first deposit up to $200', 10, None, None, None, '2026-05-31'),
        ('promo-002', 'Free Bet Friday', 'Place 3 bets on Friday, get a $10 free bet', None, 1.50, None, None, '2026-04-30'),
        ('promo-003', 'Acca Insurance', '5+ leg accumulator insurance - stake back if one leg loses', None, None, 5, None, '2026-06-30'),
        ('promo-004', 'Enhanced Odds', 'Arsenal to win at 5.00 (was 2.10) - max stake $20', None, None, None, 'evt-001', '2026-04-25'),
    ]
)
print("Seeded 4 promotions")

In [ ]:
# --- Transactions ---
cur.executemany(
    """INSERT INTO transactions (transaction_id, account_id, type, amount, method, status, created_at, description)
       VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
    [
        ('txn-001', 'default', 'deposit', 500.00, 'credit_card', 'completed', '2025-11-15T09:05:00Z', 'Initial deposit'),
        ('txn-002', 'default', 'deposit', 300.00, 'bank_transfer', 'completed', '2026-01-10T12:00:00Z', 'Bank transfer deposit'),
        ('txn-003', 'default', 'bonus', 200.00, 'system', 'completed', '2025-11-15T09:06:00Z', 'Welcome bonus credited'),
        ('txn-004', 'default', 'withdrawal', -150.00, 'bank_transfer', 'completed', '2026-03-01T10:00:00Z', 'Withdrawal to bank'),
        ('txn-005', 'default', 'bet_settlement', 150.00, 'system', 'completed', '2026-04-20T18:00:00Z', 'Winning bet payout - evt-020'),
        ('txn-010', 'cust-002', 'deposit', 200.00, 'credit_card', 'completed', '2026-01-20T14:05:00Z', 'Initial deposit'),
        ('txn-011', 'cust-002', 'deposit', 100.00, 'credit_card', 'completed', '2026-04-18T09:00:00Z', 'Top-up deposit'),
        ('txn-012', 'cust-002', 'bet_settlement', -50.00, 'system', 'completed', '2026-04-19T20:00:00Z', 'Lost bet - evt-003'),
        ('txn-020', 'cust-003', 'deposit', 200.00, 'credit_card', 'completed', '2026-04-22T10:05:00Z', 'Initial deposit'),
    ]
)
print("Seeded 9 transactions")

# --- Support Tickets ---
cur.executemany(
    """INSERT INTO support_tickets (ticket_id, account_id, subject, description, status, priority, category, created_at, updated_at)
       VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)""",
    [
        ('tkt-001', 'cust-002', 'Free Bet Friday not received',
         'I placed 3 bets last Friday but did not receive my free bet.',
         'open', 'medium', 'promotions', '2026-04-21T10:00:00Z', '2026-04-21T10:00:00Z'),
        ('tkt-002', 'cust-003', 'Welcome bonus not applied',
         'I deposited $200 yesterday but the welcome bonus has not appeared in my account.',
         'open', 'high', 'bonus', '2026-04-23T08:00:00Z', '2026-04-23T08:00:00Z'),
        ('tkt-003', 'default', 'Wrong bet selection',
         'I accidentally bet on the wrong selection for Arsenal vs Chelsea. Can I change it?',
         'open', 'medium', 'betting', '2026-04-23T09:30:00Z', '2026-04-23T09:30:00Z'),
    ]
)
print("Seeded 3 support tickets")

## Step 6: Verify\n<hr>

In [ ]:
print("=" * 60)
print("Verification")
print("=" * 60)

tables = ['sports', 'events', 'odds', 'accounts', 'bets', 'transactions', 'promotions', 'support_tickets']
for t in tables:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    count = cur.fetchone()[0]
    print(f"  {t:25s} -> {count} rows")

print("\n--- Sample: Football events ---")
cur.execute("SELECT id, home, away, status FROM events WHERE sport_id = 'football'")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]} vs {row[2]} ({row[3]})")

print("\n--- Sample: Live events ---")
cur.execute("SELECT id, sport_id, home, away FROM events WHERE status = 'live'")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[2]} vs {row[3]} [{row[1]}]")

print("\n--- Sample: Customer accounts ---")
cur.execute("SELECT account_id, name, email, balance, tier, kyc_status FROM accounts")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]} ({row[2]}) - ${row[3]} - tier={row[4]} kyc={row[5]}")

print("\n--- Sample: Odds for Arsenal vs Chelsea ---")
cur.execute("SELECT selection, odds FROM odds WHERE event_id = 'evt-001' ORDER BY selection")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]}")

print("\n--- Connection string for reference ---")
print(f"  postgresql://{DB_USERNAME}:{DB_PASSWORD}@{db_endpoint}:{db_port}/{DB_NAME}")

print("\nAll good! RDS database is ready.")

## Clean Up (run only when done with the demo)\n<hr>

In [ ]:
# # Uncomment to delete the RDS instance, subnet group, security group, and VPC
# cur.close()
# conn.close()
# 
# rds.delete_db_instance(
#     DBInstanceIdentifier=DB_INSTANCE_ID,
#     SkipFinalSnapshot=True,
#     DeleteAutomatedBackups=True,
# )
# print(f"Deleting RDS instance: {DB_INSTANCE_ID}")
# 
# waiter = rds.get_waiter('db_instance_deleted')
# waiter.wait(DBInstanceIdentifier=DB_INSTANCE_ID)
# print("RDS instance deleted.")
# 
# rds.delete_db_subnet_group(DBSubnetGroupName=DB_SUBNET_GROUP)
# print(f"DB subnet group {DB_SUBNET_GROUP} deleted.")
# 
# ec2.delete_security_group(GroupId=sg_id)
# print(f"Security group {sg_id} deleted.")
# 
# for sid in subnet_ids:
#     ec2.delete_subnet(SubnetId=sid)
#     print(f"Subnet {sid} deleted.")
# 
# ec2.delete_vpc(VpcId=vpc_id)
# print(f"VPC {vpc_id} deleted.")